# D5 Climate Workflow Notebook

Sectioned climate workflow with connector/fallback ingestion and simple/complex analysis outputs.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt

OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_LIVE_API = os.getenv('GEOPROMPT_ALLOW_LIVE_API', '0') == '1'


def fetch_json(url: str, fallback: dict) -> dict:
    if not ALLOW_LIVE_API:
        return fallback
    try:
        req = Request(url, headers={'User-Agent': 'geoprompt-section-d-notebook/2.0'})
        with urlopen(req, timeout=10) as response:  # nosec B310
            return json.loads(response.read().decode('utf-8'))
    except (URLError, TimeoutError, ValueError):
        return fallback


import geoprompt as gp
from geoprompt import GeoPromptFrame, frame_to_geojson, write_geojson
from geoprompt import equations as eq
from geoprompt.tools import build_scenario_report, export_scenario_report

## Section A: Pull Climate Data Sources


In [3]:
noaa = fetch_json('https://www.ncei.noaa.gov/cdo-web/api/v2/datasets', {'results': []})
nasa = fetch_json('https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M&community=RE&longitude=-111.89&latitude=40.76&start=20240101&end=20240107&format=JSON', {'properties': {'parameter': {}}})
era5 = fetch_json('https://cds.climate.copernicus.eu/api/v2/resources/reanalysis-era5-single-levels', {'status': 'offline-fallback'})
print('NOAA dataset count:', len(noaa.get('results', [])))
print('NASA parameter keys:', list(nasa.get('properties', {}).get('parameter', {}).keys()))
print('ERA5 status:', era5.get('status', 'n/a'))


NOAA dataset count: 0
NASA parameter keys: []
ERA5 status: offline-fallback


## Section B: Simple Track


In [ ]:
RAW_ZONES = [
    {'zone': 'north',   'heat_days': 18, 'flood_events': 2, 'drought_index': 0.41,
     'geometry': {'type': 'Point', 'coordinates': [-112.1, 41.2]}},
    {'zone': 'central', 'heat_days': 24, 'flood_events': 3, 'drought_index': 0.48,
     'geometry': {'type': 'Point', 'coordinates': [-111.9, 40.8]}},
    {'zone': 'south',   'heat_days': 29, 'flood_events': 4, 'drought_index': 0.56,
     'geometry': {'type': 'Point', 'coordinates': [-111.7, 40.3]}},
    {'zone': 'east',    'heat_days': 21, 'flood_events': 2, 'drought_index': 0.52,
     'geometry': {'type': 'Point', 'coordinates': [-111.4, 40.9]}},
    {'zone': 'west',    'heat_days': 26, 'flood_events': 3, 'drought_index': 0.44,
     'geometry': {'type': 'Point', 'coordinates': [-112.3, 40.7]}},
]
MAX_HEAT = max(r['heat_days'] for r in RAW_ZONES)
MAX_FLOOD = max(r['flood_events'] for r in RAW_ZONES)
MAX_DROUGHT = max(r['drought_index'] for r in RAW_ZONES)

# Compute risk scores from plain dicts — no pandas Scalar type issues.
enriched_zones = []
for row in RAW_ZONES:
    risk = round(
        (row['heat_days'] / MAX_HEAT) * 0.4
        + (row['flood_events'] / MAX_FLOOD) * 0.3
        + (row['drought_index'] / MAX_DROUGHT) * 0.3,
        4,
    )
    enriched_zones.append({**row, 'risk_score': risk})

# Build GeoPromptFrame — native spatial unit.
zones_frame = GeoPromptFrame(enriched_zones, geometry_column='geometry', crs='EPSG:4326')

# Spatial analysis: nearest climate zone neighbors.
neighbors = zones_frame.nearest_neighbors(id_column='zone', k=1, distance_method='haversine')
print('--- Climate zone risk summary ---')
print(json.dumps(zones_frame.summary(), indent=2, default=str))
print('\n--- Nearest neighbor climate zones ---')
for nb in neighbors:
    print(f"  {nb['origin']} → {nb['neighbor']}  dist={nb['distance']:.5f}°")

# Write GeoJSON using GeoPrompt.
risk_geojson_path = OUTPUT_DIR / 'd5-notebook-risk-map.geojson'
write_geojson(risk_geojson_path, zones_frame)
print(f'\nWrote GeoJSON map: {risk_geojson_path}')

# ── Inline climate risk map ───────────────────────────────────────────────────
records = zones_frame.to_records()
lons = [float(r['geometry']['coordinates'][0]) for r in records]
lats = [float(r['geometry']['coordinates'][1]) for r in records]
risks = [float(r['risk_score']) for r in records]
zones_labels = [str(r['zone']) for r in records]
heat_days = [int(r['heat_days']) for r in records]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(lons, lats, c=risks, cmap='hot_r', s=200, edgecolors='#333', zorder=5)
for lon, lat, zone, risk in zip(lons, lats, zones_labels, risks):
    axes[0].annotate(f'{zone}\n({risk:.2f})', (lon, lat), textcoords='offset points',
                     xytext=(5, 5), fontsize=9)
plt.colorbar(sc, ax=axes[0], label='Climate Risk Score')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_title('D5 Climate Risk Map by Zone')
axes[0].grid(True, alpha=0.3)

sorted_records = sorted(records, key=lambda r: -float(r['risk_score']))
bar_labels = [str(r['zone']) for r in sorted_records]
bar_values = [float(r['risk_score']) for r in sorted_records]
bar_heat = [int(r['heat_days']) for r in sorted_records]
bar_colors = ['#c0392b' if v > 0.8 else '#e67e22' if v > 0.65 else '#27ae60' for v in bar_values]
x = range(len(bar_labels))
width = 0.38
axes[1].bar([i - width/2 for i in x], bar_values, width=width, color=bar_colors, label='Risk Score')
ax2 = axes[1].twinx()
ax2.bar([i + width/2 for i in x], bar_heat, width=width, color='#60a5fa', alpha=0.7, label='Heat Days')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(bar_labels)
axes[1].set_ylabel('Risk Score')
ax2.set_ylabel('Heat Days')
axes[1].set_title('Climate Risk and Heat Days by Zone')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')
axes[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

simple_payload = {
    'track': 'simple',
    'zone_risk': records,
    'risk_map_geojson': str(risk_geojson_path),
}
(OUTPUT_DIR / 'd5-notebook-simple.json').write_text(json.dumps(simple_payload, indent=2, default=str), encoding='utf-8')
print('Wrote d5-notebook-simple.json')

GeoPrompt map feature count: 5


,zone,heat_days,flood_events,drought_index,lon,lat,risk_score
2,south,29,4,0.56,-111.7,40.3,1.0000
4,west,26,3,0.44,-112.3,40.7,0.8193
1,central,24,3,0.48,-111.9,40.8,0.8132
3,east,21,2,0.52,-111.4,40.9,0.7182
0,north,18,2,0.41,-112.1,41.2,0.6179


Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d5-notebook-simple.json
Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d5-notebook-risk-map.geojson
Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d5-notebook-risk-map.html


## Section C: Complex Track


In [ ]:
baseline = {'annual_loss_musd': 58.0, 'resilience_index': 0.44, 'service_reliability': 0.61}
adaptation = {'annual_loss_musd': 37.0, 'resilience_index': 0.68, 'service_reliability': 0.76}
report = build_scenario_report(baseline, adaptation, higher_is_better=['resilience_index', 'service_reliability'])
report_path = export_scenario_report(report, OUTPUT_DIR / 'd5-notebook-scenario-report.json')
print('Scenario report:', report_path)

confidence_proxy = eq.prompt_decay(1.8, scale=2.5, power=1.4)
print(f'Confidence proxy (decay): {confidence_proxy:.4f}')

# Build GeoPromptFrame for adaptation uplift analysis.
ADAPTATION_RAW = [
    {'zone': 'north',   'risk_before': 0.64, 'risk_after': 0.46,
     'geometry': {'type': 'Point', 'coordinates': [-112.1, 41.2]}},
    {'zone': 'central', 'risk_before': 0.78, 'risk_after': 0.55,
     'geometry': {'type': 'Point', 'coordinates': [-111.9, 40.8]}},
    {'zone': 'south',   'risk_before': 0.83, 'risk_after': 0.59,
     'geometry': {'type': 'Point', 'coordinates': [-111.7, 40.3]}},
    {'zone': 'east',    'risk_before': 0.67, 'risk_after': 0.47,
     'geometry': {'type': 'Point', 'coordinates': [-111.4, 40.9]}},
    {'zone': 'west',    'risk_before': 0.74, 'risk_after': 0.52,
     'geometry': {'type': 'Point', 'coordinates': [-112.3, 40.7]}},
]
adaptation_enriched = []
for row in ADAPTATION_RAW:
    delta = round(float(row['risk_before']) - float(row['risk_after']), 4)
    adaptation_enriched.append({**row, 'risk_delta': delta})

uplift_frame = GeoPromptFrame(adaptation_enriched, geometry_column='geometry', crs='EPSG:4326')

# Spatial analysis: cross-zone query for zones with high uplift.
best_delta = max(float(r['risk_delta']) for r in uplift_frame.to_records())
anchor_zone = next(str(r['zone']) for r in uplift_frame.to_records() if float(r['risk_delta']) == best_delta)
nearby = uplift_frame.query_radius(anchor=anchor_zone, max_distance=0.7, id_column='zone')
print(f'\nBest uplift anchor: {anchor_zone}  delta={best_delta:.3f}')
print(f'Zones within 0.7° of best anchor: {[str(r["zone"]) for r in nearby.to_records()]}')

# Write GeoJSON uplift map using GeoPrompt.
uplift_geojson_path = OUTPUT_DIR / 'd5-notebook-adaptation-uplift.geojson'
write_geojson(uplift_geojson_path, uplift_frame)
print(f'\nWrote uplift GeoJSON: {uplift_geojson_path}')

# ── Inline adaptation uplift charts ──────────────────────────────────────────
uplift_records = uplift_frame.to_records()
zone_names = [str(r['zone']) for r in uplift_records]
risk_before = [float(r['risk_before']) for r in uplift_records]
risk_after = [float(r['risk_after']) for r in uplift_records]
risk_deltas = [float(r['risk_delta']) for r in uplift_records]
lons_u = [float(r['geometry']['coordinates'][0]) for r in uplift_records]
lats_u = [float(r['geometry']['coordinates'][1]) for r in uplift_records]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Before vs after scatter map.
sc = axes[0].scatter(lons_u, lats_u, c=risk_after, cmap='RdYlGn_r', s=200, edgecolors='#333', zorder=5)
for lon, lat, zone, rb, ra in zip(lons_u, lats_u, zone_names, risk_before, risk_after):
    axes[0].annotate(f'{zone}\n{rb:.2f}→{ra:.2f}', (lon, lat), textcoords='offset points',
                     xytext=(5, 5), fontsize=8)
plt.colorbar(sc, ax=axes[0], label='Risk After Adaptation')
axes[0].set_title('Adaptation Risk (before→after)')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].grid(True, alpha=0.3)

# Grouped bar: before vs after.
x = range(len(zone_names))
width = 0.38
axes[1].bar([i - width/2 for i in x], risk_before, width=width, label='Before', color='#e74c3c')
axes[1].bar([i + width/2 for i in x], risk_after, width=width, label='After', color='#27ae60')
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(zone_names)
axes[1].set_ylabel('Risk Score')
axes[1].set_title('Risk Before vs After Adaptation')
axes[1].legend()
axes[1].grid(True, axis='y', alpha=0.3)

# Uplift delta bar.
delta_colors = ['#27ae60' if d > 0.2 else '#f39c12' for d in risk_deltas]
axes[2].barh(zone_names, risk_deltas, color=delta_colors)
axes[2].axvline(0.18, color='#555', linestyle='--', alpha=0.7, label='Target reduction')
axes[2].set_xlabel('Risk Reduction (delta)')
axes[2].set_title('Adaptation Uplift by Zone')
axes[2].legend()
axes[2].grid(True, axis='x', alpha=0.3)

plt.suptitle('D5 Climate — Adaptation Scenario Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

# ── Scenario comparison chart ─────────────────────────────────────────────────
metrics = list(baseline.keys())
baseline_vals = [float(baseline[m]) for m in metrics]
adapt_vals = [float(adaptation[m]) for m in metrics]

fig2, ax = plt.subplots(figsize=(9, 4))
x = range(len(metrics))
width = 0.38
ax.bar([i - width/2 for i in x], baseline_vals, width=width, label='Baseline', color='#94a3b8')
ax.bar([i + width/2 for i in x], adapt_vals, width=width, label='Adaptation', color='#2563eb')
ax.set_xticks(list(x))
ax.set_xticklabels(metrics, rotation=10)
ax.set_title('D5 Baseline vs Adaptation Portfolio Metrics')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

complex_payload = {
    'track': 'complex',
    'confidence_proxy': confidence_proxy,
    'scenario_report_path': str(report_path),
    'uplift_map_geojson': str(uplift_geojson_path),
}
(OUTPUT_DIR / 'd5-notebook-complex.json').write_text(json.dumps(complex_payload, indent=2, default=str), encoding='utf-8')
print('Wrote d5-notebook-complex.json')

GeoPrompt uplift features: 5


,zone,lon,lat,risk_before,risk_after,risk_delta
0,north,-112.1,41.2,0.64,0.46,0.18
1,central,-111.9,40.8,0.78,0.55,0.23
2,south,-111.7,40.3,0.83,0.59,0.24
3,east,-111.4,40.9,0.67,0.47,0.20
4,west,-112.3,40.7,0.74,0.52,0.22


Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d5-notebook-complex.json
Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d5-notebook-adaptation-uplift.geojson
Wrote d:\Github\geoprompt\examples\notebooks\section_d\outputs\d5-notebook-adaptation-uplift.html
